In [11]:
#!/usr/bin/env python

import os
# Silence icechunk rust warnings. Must be set before importing icechunk.
os.environ.setdefault("RUST_LOG", "error")

import icechunk
from virtualizarr.parsers.hdf.hdf import _construct_manifest_array

import obstore
from obstore.store import HTTPStore

import shutil
import io
from pathlib import Path
import logging

import numpy as np
import h5py
import zarr
from zarr.codecs import Zlib
import xarray as xr
import pandas as pd

from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# constants
NLAT, NLON = 1800, 3600          # HG grid dimensions
CLAT       = 72                  # latitude chunk size (matches source HDF5)

# Fill values (raw int16, before scale factor)
FILL_MISSING  = np.int16(-32768)   # cloudy / no retrieval
FILL_NOOBS    = np.int16(-32767)   # ocean / no observation

SCALE_FACTOR  = 0.1               # raw × 0.1 → cm

GPORTAL_BASE = (
    "https://gportal.jaxa.jp/download/standard/GCOM-W/GCOM-W.AMSR2"
    "/L3.SND_10/2/{yyyy}/{mm}/"
)
FNAME_TMPL   = "GW1AM2_{date}_01D_{orbit}_L3SGSNDHG2210210.h5"
ORBIT_CODES  = ["EQMA",        "EQMD"       ]
ORBIT_LABELS = ["Ascending",   "Descending" ]

STORE_URL = "s3://"
STORE_DIR = Path("/tmp/amsr2-store-multi").resolve()

# CF time reference
TIME_UNITS    = "days since 1970-01-01"
TIME_CALENDAR = "proleptic_gregorian"
TIME_EPOCH    = pd.Timestamp("1970-01-01")

# helpers

def gportal_url(date: pd.Timestamp, orbit: str) -> str:
    """Return the G-Portal HTTPS URL for a given date (pd.Timestamp) and orbit code."""
    yyyy = f"{date.year:04d}"
    mm = f"{date.month:02d}"
    return GPORTAL_BASE.format(yyyy=yyyy, mm=mm) + FNAME_TMPL.format(
        date=date.strftime("%Y%m%d"), orbit=orbit
    )


def _stream_hdf5(path: str, access_options: dict = {}):
    from urllib.parse import urlsplit

    parts = urlsplit(path)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url, **access_options)
    data = obstore.get(store, obj_path).bytes()
    buf = io.BytesIO(bytes(data))

    class _CM:
        def __enter__(_):
            return buf

        def __exit__(_, *__):
            buf.close()

    return _CM()


def _av(v):
    """Convert an HDF5 attribute value to a JSON-serialisable Python scalar."""
    if isinstance(v, np.ndarray):
        flat = v.flatten()
        if flat.dtype.kind in ("S", "U", "O"):
            items = [x.decode() if isinstance(x, bytes) else str(x) for x in flat]
            return items[0] if len(items) == 1 else items
        lst = flat.tolist()
        return lst[0] if len(lst) == 1 else lst
    return v.decode() if isinstance(v, bytes) else v

def _get_manifests(chunk_url: str) -> tuple:
    """
    Stream the HDF5 file at *chunk_url* into memory and extract the
    VirtualiZarr manifest for the ``Geophysical Data`` group.

    Uses ``obstore`` to download the entire file in one HTTP request
    (``obstore_stream`` strategy), which is much faster than lazy fsspec
    streaming for HDF5 files that are not cloud-optimised.

    Returns (manifest, attrs) for the ``Geophysical Data`` group.
    Band 0 = snow depth, band 1 = quality flag.  The source array has shape
    (lat, lon, band) with chunks (CLAT, NLON, 2), so each chunk covers both
    bands.
    """
    from urllib.parse import urlsplit
    parts    = urlsplit(chunk_url)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store    = HTTPStore.from_url(base_url)
    buf      = io.BytesIO(obstore.get(store, obj_path).bytes())
    with h5py.File(buf, "r") as f:
        ma    = _construct_manifest_array(chunk_url, f["Geophysical Data"], "/")
        attrs = {k: _av(v) for k, v in f["Geophysical Data"].attrs.items()}
    return ma.manifest, attrs


def initialize_store(session: icechunk.Session, commit_msg: str = "Initialize empty store") -> None:
    """
    Create a fresh IceChunk repository at *store_dir*.

    Parameters
    ----------
    store_dir : Path
        Directory for the IceChunk repository.
    """

    lats = np.linspace(89.95, -89.95, NLAT).astype("float32")
    lons = np.linspace(0.05, 359.95, NLON).astype("float32")

    root = zarr.open_group(session.store, mode="w", zarr_format=3)

    # ── coordinate arrays ─────────────────────────────────────────────────────
    root.require_array(
        "time",
        shape=(0,),
        chunks=(512,),
        dtype="int32",
        dimension_names=["time"],
        attributes={
            "units": TIME_UNITS,
            "calendar": TIME_CALENDAR,
            "long_name": "observation date",
        },
    )
    root.require_array(
        "orbit",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["orbit"],
        attributes={"long_name": "orbit direction"},
    )
    root["orbit"][:] = ORBIT_LABELS

    root.require_array(
        "band",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["band"],
        attributes={"long_name": "band name"},
    )
    root["band"][:] = ["snow_depth", "quality_flag"]

    root.require_array(
        "lat",
        shape=(NLAT,),
        chunks=(NLAT,),
        dtype="float32",
        dimension_names=["lat"],
        attributes={"units": "degrees_north", "long_name": "latitude"},
    )
    root["lat"][:] = lats

    root.require_array(
        "lon",
        shape=(NLON,),
        chunks=(NLON,),
        dtype="float32",
        dimension_names=["lon"],
        attributes={"units": "degrees_east", "long_name": "longitude"},
    )
    root["lon"][:] = lons

    # ── data variable ─────────────────────────────────────────────────────────
    # Zlib(level=9) matches the zlib/deflate compression in the source HDF5 files.
    # The band dimension (size 2) is interleaved within each source chunk, so
    # we keep it as the innermost dimension to match the physical layout.
    root.require_array(
        "geophysical_data",
        shape=(0, 2, NLAT, NLON, 2),
        chunks=(1, 1, CLAT, NLON, 2),
        dtype="int16",
        fill_value=int(FILL_MISSING),
        compressors=[Zlib(level=9)],
        dimension_names=["time", "orbit", "lat", "lon", "band"],
        attributes={
            "long_name": "geophysical data (band 0 = snow depth, band 1 = quality flag)",
            "units": "cm",
            "scale_factor": SCALE_FACTOR,
            "_FillValue": int(FILL_MISSING),
            "missing_value": int(FILL_NOOBS),
        },
    )

    session.commit(commit_msg)
    log.info("Initialized empty store")


def stored_days(repo: icechunk.Repository) -> list[int]:
    """Return the list of int32 day values currently in the store (insertion order)."""
    session = repo.readonly_session("main")
    root    = zarr.open_group(session.store, mode="r", zarr_format=3)
    n = root["time"].shape[0]
    return list(root["time"][:].tolist()) if n > 0 else []


def insert_date(date: pd.Timestamp, repo: icechunk.Repository) -> None:
    """
    Insert *date* (YYYYMMDD) into the IceChunk store.

    Downloads each orbit's HDF5 file from G-Portal HTTPS into memory,
    extracts chunk byte offsets via VirtualiZarr, and writes virtual
    chunk references pointing back to the same HTTPS URLs.
    No data is copied; the store only records byte positions.
    Returns a human-readable status string.
    """
    day_val  = int((date - TIME_EPOCH).days)
    date_iso = date.strftime("%Y-%m-%d")

    current = stored_days(repo)

    if day_val in current:
        slot = current.index(day_val)
        log.info(f"{date_iso}  →  skipped (already present, slot {slot})")
        return

    # Build manifests for each orbit by streaming the HDF5 from G-Portal.
    orbit_manifests: dict[int, object] = {}
    for o_idx, orbit in enumerate(ORBIT_CODES):
        chunk_url = gportal_url(date, orbit)
        try:
            manifest, _ = _get_manifests(chunk_url)
            orbit_manifests[o_idx] = manifest
        except Exception as exc:
            log.warning(f"  could not fetch {orbit} for {date_iso}: {exc}")

    if not orbit_manifests:
        log.info(f"{date_iso}  →  skipped (no orbits available on G-Portal)")
        return

    new_slot = len(current)

    session = repo.writable_session("main")
    store   = session.store
    root    = zarr.open_group(store, mode="r+", zarr_format=3)

    # Grow time dimension and data array
    root["time"].resize((new_slot + 1,))
    root["time"][new_slot] = day_val
    root["geophysical_data"].resize((new_slot + 1, 2, NLAT, NLON, 2))

    # Write virtual chunk refs.
    # Source chunks have shape (CLAT, NLON, 2) — both bands in one chunk.
    # The manifest key (ci, cj, ck) maps to (lat-chunk, lon-chunk, band-chunk).
    # Since chunk size covers the full band dim, ck is always 0.
    for o_idx, manifest in orbit_manifests.items():
        for (ci, cj, ck), ref in manifest.iter_refs():
            key = f"geophysical_data/c/{new_slot}/{o_idx}/{ci}/{cj}/{ck}"
            store.set_virtual_ref(
                key, ref["path"], offset=ref["offset"], length=ref["length"]
            )

    session.commit(f"Insert {date_iso} → slot {new_slot}")

################################################################################

# Initialize icechunk store
jaxa_gportal_url = "https://gportal.jaxa.jp/"

try:
    repo = icechunk.Repository.open(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND"
        ),
        authorize_virtual_chunk_access={jaxa_gportal_url: None}
    )
    session = repo.writable_session("main")
    log.info("Found existing icechunk store")
except icechunk.IcechunkError as e:
    log.warning(
        f"Failed to open existing repo with error: {str(e)}. "
        "Trying to create a fresh repo."
    )
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(jaxa_gportal_url, icechunk.http_store())
    )
    repo = icechunk.Repository.create(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND"
        ),
        config=config,
        authorize_virtual_chunk_access={jaxa_gportal_url: None}
    )
    session = repo.writable_session("main")
    initialize_store(session)

date_seq = pd.date_range(start="2018-09-01", end="2019-07-01", freq="D")

for date in tqdm(date_seq):
    insert_date(date, repo)

################################################################################

log.info("Testing reading a subset of data")
# Try reading the data
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store)

sub = ds.sel(lat=52.07, lon=-91.99, method="nearest").sel(
    orbit="Descending", band="snow_depth", time=slice("2018-10-01", "2019-02-01"))
print(sub["geophysical_data"].values)

/tmp/ipykernel_450/3936817360.py:289: DeprecationWarning: Passing `None` in `authorize_virtual_chunk_access` for container `https://gportal.jaxa.jp/` is deprecated and will be unsupported in a future release; pass an explicit credential or no-auth sentinel instead. For example:
    authorize_virtual_chunk_access={"https://gportal.jaxa.jp/": ic.credentials.HttpAccess} See https://github.com/earth-mover/icechunk/issues/2194 for details.
  repo = icechunk.Repository.open(
  2026-08-14T19:25:36.832666Z  WARN icechunk::repository: DEPRECATED: passing `None` to authorize virtual chunk access for container `https://gportal.jaxa.jp/` is deprecated and will be rejected in a future release. Pass the explicit `Credentials::HttpAccess` sentinel instead., url_prefix: "https://gportal.jaxa.jp/"
    at icechunk/src/repository.rs:2224

INFO:__main__:Found existing icechunk store
  0%|          | 0/304 [00:00<?, ?it/s]INFO:__main__:2018-09-01  →  skipped (already present, slot 0)
INFO:__main__:2018-09-

KeyError: 'Value based partial slicing on non-monotonic DatetimeIndexes with non-existing keys is not allowed.'

In [13]:
#!/usr/bin/env python

import os
# Silence icechunk rust warnings. Must be set before importing icechunk.
os.environ.setdefault("RUST_LOG", "error")

import icechunk
from virtualizarr.parsers.hdf.hdf import _construct_manifest_array

import obstore
from obstore.store import HTTPStore

import io
import logging
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from zarr.codecs import Zlib
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# Constants
NLAT, NLON = 1800, 3600          # HG grid dimensions
CLAT = 72                        # latitude chunk size (matches source HDF5)

# Fill values (raw int16, before scale factor)
FILL_MISSING = np.int16(-32768)  # cloudy / no retrieval
FILL_NOOBS = np.int16(-32767)    # ocean / no observation

SCALE_FACTOR = 0.1               # raw × 0.1 → cm

GPORTAL_BASE = (
    "https://gportal.jaxa.jp/download/standard/GCOM-W/GCOM-W.AMSR2"
    "/L3.SND_10/2/{yyyy}/{mm}/"
)
FNAME_TMPL = "GW1AM2_{date}_01D_{orbit}_L3SGSNDHG2210210.h5"
ORBIT_CODES = ["EQMA", "EQMD"]
ORBIT_LABELS = ["Ascending", "Descending"]

# CF time reference
TIME_UNITS = "days since 1970-01-01"
TIME_CALENDAR = "proleptic_gregorian"
TIME_EPOCH = pd.Timestamp("1970-01-01")

GPORTAL_URL = "https://gportal.jaxa.jp/"


def gportal_url(date: pd.Timestamp, orbit: str) -> str:
    """Return the G-Portal HTTPS URL for a date and orbit code."""
    return GPORTAL_BASE.format(
        yyyy=f"{date.year:04d}", mm=f"{date.month:02d}"
    ) + FNAME_TMPL.format(
        date=date.strftime("%Y%m%d"), orbit=orbit
    )


def _stream_hdf5(path: str, access_options: dict | None = None):
    """Return a context manager containing an HDF5 HTTP object in memory."""
    from urllib.parse import urlsplit

    if access_options is None:
        access_options = {}

    parts = urlsplit(path)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url, **access_options)
    data = obstore.get(store, obj_path).bytes()
    buf = io.BytesIO(bytes(data))

    class _CM:
        def __enter__(self):
            return buf

        def __exit__(self, *_):
            buf.close()

    return _CM()


def _av(value):
    """Convert an HDF5 attribute value to a JSON-serialisable value."""
    if isinstance(value, np.ndarray):
        flat = value.flatten()
        if flat.dtype.kind in ("S", "U", "O"):
            items = [
                item.decode() if isinstance(item, bytes) else str(item)
                for item in flat
            ]
            return items[0] if len(items) == 1 else items
        values = flat.tolist()
        return values[0] if len(values) == 1 else values
    return value.decode() if isinstance(value, bytes) else value


def _get_manifests(chunk_url: str) -> tuple:
    """Download an HDF5 file and extract its VirtualiZarr manifest."""
    from urllib.parse import urlsplit

    parts = urlsplit(chunk_url)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url)
    buf = io.BytesIO(obstore.get(store, obj_path).bytes())

    try:
        with h5py.File(buf, "r") as hdf:
            geophysical_data = hdf["Geophysical Data"]
            manifest_array = _construct_manifest_array(
                chunk_url, geophysical_data, "/"
            )
            attrs = {
                key: _av(value)
                for key, value in geophysical_data.attrs.items()
            }
        return manifest_array.manifest, attrs
    finally:
        buf.close()


def initialize_store(session: icechunk.Session,
                     commit_msg: str = "Initialize empty store") -> None:
    """Create the arrays in a new IceChunk repository."""
    lats = np.linspace(89.95, -89.95, NLAT).astype("float32")
    lons = np.linspace(0.05, 359.95, NLON).astype("float32")

    root = zarr.open_group(session.store, mode="w", zarr_format=3)

    root.require_array(
        "time",
        shape=(0,),
        chunks=(512,),
        dtype="int32",
        dimension_names=["time"],
        attributes={
            "units": TIME_UNITS,
            "calendar": TIME_CALENDAR,
            "long_name": "observation date",
        },
    )

    root.require_array(
        "orbit",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["orbit"],
        attributes={"long_name": "orbit direction"},
    )
    root["orbit"][:] = ORBIT_LABELS

    root.require_array(
        "band",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["band"],
        attributes={"long_name": "band name"},
    )
    root["band"][:] = ["snow_depth", "quality_flag"]

    root.require_array(
        "lat",
        shape=(NLAT,),
        chunks=(NLAT,),
        dtype="float32",
        dimension_names=["lat"],
        attributes={
            "units": "degrees_north",
            "long_name": "latitude",
        },
    )
    root["lat"][:] = lats

    root.require_array(
        "lon",
        shape=(NLON,),
        chunks=(NLON,),
        dtype="float32",
        dimension_names=["lon"],
        attributes={
            "units": "degrees_east",
            "long_name": "longitude",
        },
    )
    root["lon"][:] = lons

    root.require_array(
        "geophysical_data",
        shape=(0, 2, NLAT, NLON, 2),
        chunks=(1, 1, CLAT, NLON, 2),
        dtype="int16",
        fill_value=int(FILL_MISSING),
        compressors=[Zlib(level=9)],
        dimension_names=["time", "orbit", "lat", "lon", "band"],
        attributes={
            "long_name": (
                "geophysical data (band 0 = snow depth, "
                "band 1 = quality flag)"
            ),
            "units": "cm",
            "scale_factor": SCALE_FACTOR,
            "_FillValue": int(FILL_MISSING),
            "missing_value": int(FILL_NOOBS),
        },
    )

    session.commit(commit_msg)
    log.info("Initialized empty store")


def stored_days(repo: icechunk.Repository) -> list[int]:
    """Return stored day values in their current storage order."""
    session = repo.readonly_session("main")
    root = zarr.open_group(session.store, mode="r", zarr_format=3)
    return root["time"][:].tolist()


def insert_date(date: pd.Timestamp, repo: icechunk.Repository) -> None:
    """Insert a date using virtual references to the source HDF5 chunks."""
    day_val = int((date - TIME_EPOCH).days)
    date_iso = date.strftime("%Y-%m-%d")
    current = stored_days(repo)

    if day_val in current:
        slot = current.index(day_val)
        log.info("%s → skipped (already present, slot %d)", date_iso, slot)
        return

    orbit_manifests: dict[int, object] = {}
    for orbit_index, orbit in enumerate(ORBIT_CODES):
        chunk_url = gportal_url(date, orbit)
        try:
            manifest, _ = _get_manifests(chunk_url)
            orbit_manifests[orbit_index] = manifest
        except Exception as exc:
            log.warning(
                "  could not fetch %s for %s: %s",
                orbit,
                date_iso,
                exc,
            )

    if not orbit_manifests:
        log.info("%s → skipped (no orbits available on G-Portal)", date_iso)
        return

    new_slot = len(current)
    session = repo.writable_session("main")
    root = zarr.open_group(session.store, mode="r+", zarr_format=3)

    root["time"].resize((new_slot + 1,))
    root["time"][new_slot] = day_val
    root["geophysical_data"].resize((new_slot + 1, 2, NLAT, NLON, 2))

    # Each source chunk has shape (CLAT, NLON, 2), so both bands are
    # represented by the same virtual source chunk.
    for orbit_index, manifest in orbit_manifests.items():
        for (lat_chunk, lon_chunk, band_chunk), reference in manifest.iter_refs():
            key = (
                f"geophysical_data/c/{new_slot}/{orbit_index}/"
                f"{lat_chunk}/{lon_chunk}/{band_chunk}"
            )
            store_key = session.store
            store_key.set_virtual_ref(
                key,
                reference["path"],
                offset=reference["offset"],
                length=reference["length"],
            )

    session.commit(f"Insert {date_iso} → slot {new_slot}")


def select_time_range(ds: xr.Dataset, start: str, end: str) -> xr.Dataset:
    """
    Select a time interval without requiring a sorted or complete time index.

    ``Dataset.sel(time=slice(...))`` requires a monotonic DatetimeIndex when
    one or both slice endpoints are absent from a non-monotonic index. Dates
    are appended to the repository, so an existing repository may not have
    monotonically ordered time values. Boolean selection avoids that pandas
    restriction and also correctly handles missing days.
    """
    times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
    mask = (times >= pd.Timestamp(start)) & (times <= pd.Timestamp(end))
    return ds.isel(time=np.flatnonzero(mask))


# Open or create the IceChunk store.
try:
    repo = icechunk.Repository.open(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        authorize_virtual_chunk_access={GPORTAL_URL: icechunk.credentials.HttpAccess},
    )
    log.info("Found existing icechunk store")
except icechunk.IcechunkError as exc:
    log.warning(
        "Failed to open existing repo with error: %s. "
        "Trying to create a fresh repo.",
        exc,
    )
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(GPORTAL_URL, icechunk.http_store())
    )
    repo = icechunk.Repository.create(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        config=config,
        authorize_virtual_chunk_access={GPORTAL_URL: icechunk.credentials.HttpAccess},
    )
    initialize_store(repo.writable_session("main"))


date_seq = pd.date_range(
    start="2018-09-01",
    end="2019-07-01",
    freq="D",
)

for date in tqdm(date_seq):
    insert_date(date, repo)


log.info("Testing reading a subset of data")
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store)

# Spatial, orbit, and band selection can use normal label selection. Time is
# selected with a boolean index because the repository may contain dates in
# insertion order rather than chronological order.
sub = ds.sel(lat=52.07, lon=-91.99, method="nearest")
sub = sub.sel(orbit="Descending", band="snow_depth")
sub = select_time_range(sub, "2018-10-01", "2019-02-01")

print(sub["geophysical_data"].values)

INFO:__main__:Found existing icechunk store
  0%|          | 1/304 [00:00<00:33,  8.96it/s]INFO:__main__:2018-09-02 → skipped (already present, slot 1)
INFO:__main__:2018-09-03 → skipped (already present, slot 2)
INFO:__main__:2018-09-04 → skipped (already present, slot 3)
  1%|▏         | 4/304 [00:00<00:17, 17.05it/s]INFO:__main__:2018-09-05 → skipped (already present, slot 4)
INFO:__main__:2018-09-06 → skipped (already present, slot 5)
  2%|▏         | 6/304 [00:00<00:16, 18.15it/s]INFO:__main__:2018-09-07 → skipped (already present, slot 6)
INFO:__main__:2018-09-08 → skipped (already present, slot 7)
INFO:__main__:2018-09-09 → skipped (already present, slot 8)
  3%|▎         | 9/304 [00:00<00:14, 19.80it/s]INFO:__main__:2018-09-10 → skipped (already present, slot 9)
INFO:__main__:2018-09-11 → skipped (already present, slot 10)
INFO:__main__:2018-09-12 → skipped (already present, slot 11)
  4%|▍         | 12/304 [00:00<00:14, 20.25it/s]INFO:__main__:2018-09-13 → skipped (already pre

[ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.  27.9 59.4  0. ]


In [14]:
#!/usr/bin/env python

import os
# Silence icechunk rust warnings. Must be set before importing icechunk.
os.environ.setdefault("RUST_LOG", "error")

import icechunk
from virtualizarr.parsers.hdf.hdf import _construct_manifest_array

import obstore
from obstore.store import HTTPStore

import io
import logging
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from zarr.codecs import Zlib
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# Constants
NLAT, NLON = 1800, 3600          # HG grid dimensions
CLAT = 72                        # latitude chunk size (matches source HDF5)

# Fill values (raw int16, before scale factor)
FILL_MISSING = np.int16(-32768)  # cloudy / no retrieval
FILL_NOOBS = np.int16(-32767)    # ocean / no observation

SCALE_FACTOR = 0.1               # raw × 0.1 → cm

GPORTAL_BASE = (
    "https://gportal.jaxa.jp/download/standard/GCOM-W/GCOM-W.AMSR2"
    "/L3.SND_10/2/{yyyy}/{mm}/"
)
FNAME_TMPL = "GW1AM2_{date}_01D_{orbit}_L3SGSNDHG2210210.h5"
ORBIT_CODES = ["EQMA", "EQMD"]
ORBIT_LABELS = ["Ascending", "Descending"]

# CF time reference
TIME_UNITS = "days since 1970-01-01"
TIME_CALENDAR = "proleptic_gregorian"
TIME_EPOCH = pd.Timestamp("1970-01-01")

GPORTAL_URL = "https://gportal.jaxa.jp/"


def gportal_url(date: pd.Timestamp, orbit: str) -> str:
    """Return the G-Portal HTTPS URL for a date and orbit code."""
    return GPORTAL_BASE.format(
        yyyy=f"{date.year:04d}", mm=f"{date.month:02d}"
    ) + FNAME_TMPL.format(
        date=date.strftime("%Y%m%d"), orbit=orbit
    )


def _stream_hdf5(path: str, access_options: dict | None = None):
    """Return a context manager containing an HDF5 HTTP object in memory."""
    from urllib.parse import urlsplit

    if access_options is None:
        access_options = {}

    parts = urlsplit(path)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url, **access_options)
    data = obstore.get(store, obj_path).bytes()
    buf = io.BytesIO(bytes(data))

    class _CM:
        def __enter__(self):
            return buf

        def __exit__(self, *_):
            buf.close()

    return _CM()


def _av(value):
    """Convert an HDF5 attribute value to a JSON-serialisable value."""
    if isinstance(value, np.ndarray):
        flat = value.flatten()
        if flat.dtype.kind in ("S", "U", "O"):
            items = [
                item.decode() if isinstance(item, bytes) else str(item)
                for item in flat
            ]
            return items[0] if len(items) == 1 else items
        values = flat.tolist()
        return values[0] if len(values) == 1 else values
    return value.decode() if isinstance(value, bytes) else value


def _get_manifests(chunk_url: str) -> tuple:
    """Download an HDF5 file and extract its VirtualiZarr manifest."""
    from urllib.parse import urlsplit

    parts = urlsplit(chunk_url)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url)
    buf = io.BytesIO(obstore.get(store, obj_path).bytes())

    try:
        with h5py.File(buf, "r") as hdf:
            geophysical_data = hdf["Geophysical Data"]
            manifest_array = _construct_manifest_array(
                chunk_url, geophysical_data, "/"
            )
            attrs = {
                key: _av(value)
                for key, value in geophysical_data.attrs.items()
            }
        return manifest_array.manifest, attrs
    finally:
        buf.close()


def initialize_store(
    session: icechunk.Session,
    commit_msg: str = "Initialize empty store",
) -> None:
    """Create the arrays in a new IceChunk repository."""
    lats = np.linspace(89.95, -89.95, NLAT).astype("float32")
    lons = np.linspace(0.05, 359.95, NLON).astype("float32")

    root = zarr.open_group(session.store, mode="w", zarr_format=3)

    root.require_array(
        "time",
        shape=(0,),
        chunks=(512,),
        dtype="int32",
        dimension_names=["time"],
        attributes={
            "units": TIME_UNITS,
            "calendar": TIME_CALENDAR,
            "long_name": "observation date",
        },
    )

    root.require_array(
        "orbit",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["orbit"],
        attributes={"long_name": "orbit direction"},
    )
    root["orbit"][:] = ORBIT_LABELS

    root.require_array(
        "band",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["band"],
        attributes={"long_name": "band name"},
    )
    root["band"][:] = ["snow_depth", "quality_flag"]

    root.require_array(
        "lat",
        shape=(NLAT,),
        chunks=(NLAT,),
        dtype="float32",
        dimension_names=["lat"],
        attributes={
            "units": "degrees_north",
            "long_name": "latitude",
        },
    )
    root["lat"][:] = lats

    root.require_array(
        "lon",
        shape=(NLON,),
        chunks=(NLON,),
        dtype="float32",
        dimension_names=["lon"],
        attributes={
            "units": "degrees_east",
            "long_name": "longitude",
        },
    )
    root["lon"][:] = lons

    root.require_array(
        "geophysical_data",
        shape=(0, 2, NLAT, NLON, 2),
        chunks=(1, 1, CLAT, NLON, 2),
        dtype="int16",
        fill_value=int(FILL_MISSING),
        compressors=[Zlib(level=9)],
        dimension_names=["time", "orbit", "lat", "lon", "band"],
        attributes={
            "long_name": (
                "geophysical data (band 0 = snow depth, "
                "band 1 = quality flag)"
            ),
            "units": "cm",
            "scale_factor": SCALE_FACTOR,
            # Keep only one CF fill/missing value.  xarray otherwise emits a
            # SerializationWarning because both _FillValue and
            # missing_value would be decoded to NaN.  The no-observation code
            # remains available in the raw data and is documented separately.
            "_FillValue": int(FILL_MISSING),
            "no_observation_value": int(FILL_NOOBS),
        },
    )

    session.commit(commit_msg)
    log.info("Initialized empty store")


def stored_days(repo: icechunk.Repository) -> list[int]:
    """Return stored day values in their current storage order."""
    session = repo.readonly_session("main")
    root = zarr.open_group(session.store, mode="r", zarr_format=3)
    return root["time"][:].tolist()


def insert_date(date: pd.Timestamp, repo: icechunk.Repository) -> None:
    """Insert a date using virtual references to the source HDF5 chunks."""
    day_val = int((date - TIME_EPOCH).days)
    date_iso = date.strftime("%Y-%m-%d")
    current = stored_days(repo)

    if day_val in current:
        slot = current.index(day_val)
        log.info("%s → skipped (already present, slot %d)", date_iso, slot)
        return

    orbit_manifests: dict[int, object] = {}
    for orbit_index, orbit in enumerate(ORBIT_CODES):
        chunk_url = gportal_url(date, orbit)
        try:
            manifest, _ = _get_manifests(chunk_url)
            orbit_manifests[orbit_index] = manifest
        except Exception as exc:
            log.warning(
                "  could not fetch %s for %s: %s",
                orbit,
                date_iso,
                exc,
            )

    if not orbit_manifests:
        log.info("%s → skipped (no orbits available on G-Portal)", date_iso)
        return

    new_slot = len(current)
    session = repo.writable_session("main")
    root = zarr.open_group(session.store, mode="r+", zarr_format=3)

    root["time"].resize((new_slot + 1,))
    root["time"][new_slot] = day_val
    root["geophysical_data"].resize((new_slot + 1, 2, NLAT, NLON, 2))

    # Each source chunk has shape (CLAT, NLON, 2), so both bands are
    # represented by the same virtual source chunk.
    for orbit_index, manifest in orbit_manifests.items():
        for (lat_chunk, lon_chunk, band_chunk), reference in manifest.iter_refs():
            key = (
                f"geophysical_data/c/{new_slot}/{orbit_index}/"
                f"{lat_chunk}/{lon_chunk}/{band_chunk}"
            )
            session.store.set_virtual_ref(
                key,
                reference["path"],
                offset=reference["offset"],
                length=reference["length"],
            )

    session.commit(f"Insert {date_iso} → slot {new_slot}")


def select_time_range(ds: xr.Dataset, start: str, end: str) -> xr.Dataset:
    """
    Select a time interval without requiring a sorted or complete time index.

    Dates are appended to the repository, so an existing repository may not
    have monotonically ordered time values. Boolean selection avoids pandas'
    restriction on slicing a non-monotonic DatetimeIndex.
    """
    times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
    mask = (times >= pd.Timestamp(start)) & (times <= pd.Timestamp(end))
    return ds.isel(time=np.flatnonzero(mask))


# Open or create the IceChunk store.
try:
    repo = icechunk.Repository.open(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        authorize_virtual_chunk_access={GPORTAL_URL: None},
    )
    log.info("Found existing icechunk store")
except icechunk.IcechunkError as exc:
    log.warning(
        "Failed to open existing repo with error: %s. "
        "Trying to create a fresh repo.",
        exc,
    )
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(GPORTAL_URL, icechunk.http_store())
    )
    repo = icechunk.Repository.create(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        config=config,
        authorize_virtual_chunk_access={GPORTAL_URL: None},
    )
    initialize_store(repo.writable_session("main"))


date_seq = pd.date_range(
    start="2018-09-01",
    end="2019-07-01",
    freq="D",
)

for date in tqdm(date_seq):
    insert_date(date, repo)


log.info("Testing reading a subset of data")
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store)

# Spatial, orbit, and band selection can use normal label selection. Time is
# selected with a boolean index because the repository may contain dates in
# insertion order rather than chronological order.
sub = ds.sel(lat=52.07, lon=-91.99, method="nearest")
sub = sub.sel(orbit="Descending", band="snow_depth")
sub = select_time_range(sub, "2018-10-01", "2019-02-01")

print(sub["geophysical_data"].values)

/tmp/ipykernel_450/2634456288.py:301: DeprecationWarning: Passing `None` in `authorize_virtual_chunk_access` for container `https://gportal.jaxa.jp/` is deprecated and will be unsupported in a future release; pass an explicit credential or no-auth sentinel instead. For example:
    authorize_virtual_chunk_access={"https://gportal.jaxa.jp/": ic.credentials.HttpAccess} See https://github.com/earth-mover/icechunk/issues/2194 for details.
  repo = icechunk.Repository.open(
  2026-08-14T19:32:53.154800Z  WARN icechunk::repository: DEPRECATED: passing `None` to authorize virtual chunk access for container `https://gportal.jaxa.jp/` is deprecated and will be rejected in a future release. Pass the explicit `Credentials::HttpAccess` sentinel instead., url_prefix: "https://gportal.jaxa.jp/"
    at icechunk/src/repository.rs:2224

INFO:__main__:Found existing icechunk store
  0%|          | 1/304 [00:00<00:49,  6.16it/s]INFO:__main__:2018-09-02 → skipped (already present, slot 1)
INFO:__main__:20

[ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.  27.9 59.4  0. ]


In [1]:
#!/usr/bin/env python

import os
# Silence icechunk rust warnings. Must be set before importing icechunk.
os.environ.setdefault("RUST_LOG", "error")

import icechunk
from virtualizarr.parsers.hdf.hdf import _construct_manifest_array

import obstore
from obstore.store import HTTPStore

import io
import logging
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from zarr.codecs import Zlib
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# Constants
NLAT, NLON = 1800, 3600          # HG grid dimensions
CLAT = 72                        # latitude chunk size (matches source HDF5)

# Fill values (raw int16, before scale factor)
FILL_MISSING = np.int16(-32768)  # cloudy / no retrieval
FILL_NOOBS = np.int16(-32767)    # ocean / no observation

SCALE_FACTOR = 0.1               # raw × 0.1 → cm

GPORTAL_BASE = (
    "https://gportal.jaxa.jp/download/standard/GCOM-W/GCOM-W.AMSR2"
    "/L3.SND_10/2/{yyyy}/{mm}/"
)
FNAME_TMPL = "GW1AM2_{date}_01D_{orbit}_L3SGSNDHG2210210.h5"
ORBIT_CODES = ["EQMA", "EQMD"]
ORBIT_LABELS = ["Ascending", "Descending"]

# CF time reference
TIME_UNITS = "days since 1970-01-01"
TIME_CALENDAR = "proleptic_gregorian"
TIME_EPOCH = pd.Timestamp("1970-01-01")

GPORTAL_URL = "https://gportal.jaxa.jp/"


def gportal_url(date: pd.Timestamp, orbit: str) -> str:
    """Return the G-Portal HTTPS URL for a date and orbit code."""
    return GPORTAL_BASE.format(
        yyyy=f"{date.year:04d}", mm=f"{date.month:02d}"
    ) + FNAME_TMPL.format(
        date=date.strftime("%Y%m%d"), orbit=orbit
    )


def _stream_hdf5(path: str, access_options: dict | None = None):
    """Return a context manager containing an HDF5 HTTP object in memory."""
    from urllib.parse import urlsplit

    if access_options is None:
        access_options = {}

    parts = urlsplit(path)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url, **access_options)
    data = obstore.get(store, obj_path).bytes()
    buf = io.BytesIO(bytes(data))

    class _CM:
        def __enter__(self):
            return buf

        def __exit__(self, *_):
            buf.close()

    return _CM()


def _av(value):
    """Convert an HDF5 attribute value to a JSON-serialisable value."""
    if isinstance(value, np.ndarray):
        flat = value.flatten()
        if flat.dtype.kind in ("S", "U", "O"):
            items = [
                item.decode() if isinstance(item, bytes) else str(item)
                for item in flat
            ]
            return items[0] if len(items) == 1 else items
        values = flat.tolist()
        return values[0] if len(values) == 1 else values
    return value.decode() if isinstance(value, bytes) else value


def _get_manifests(chunk_url: str) -> tuple:
    """Download an HDF5 file and extract its VirtualiZarr manifest."""
    from urllib.parse import urlsplit

    parts = urlsplit(chunk_url)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url)
    buf = io.BytesIO(obstore.get(store, obj_path).bytes())

    try:
        with h5py.File(buf, "r") as hdf:
            geophysical_data = hdf["Geophysical Data"]
            manifest_array = _construct_manifest_array(
                chunk_url, geophysical_data, "/"
            )
            attrs = {
                key: _av(value)
                for key, value in geophysical_data.attrs.items()
            }
        return manifest_array.manifest, attrs
    finally:
        buf.close()


def initialize_store(
    session: icechunk.Session,
    commit_msg: str = "Initialize empty store",
) -> None:
    """Create the arrays in a new IceChunk repository."""
    lats = np.linspace(89.95, -89.95, NLAT).astype("float32")
    lons = np.linspace(0.05, 359.95, NLON).astype("float32")

    root = zarr.open_group(session.store, mode="w", zarr_format=3)

    root.require_array(
        "time",
        shape=(0,),
        chunks=(512,),
        dtype="int32",
        dimension_names=["time"],
        attributes={
            "units": TIME_UNITS,
            "calendar": TIME_CALENDAR,
            "long_name": "observation date",
        },
    )

    root.require_array(
        "orbit",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["orbit"],
        attributes={"long_name": "orbit direction"},
    )
    root["orbit"][:] = ORBIT_LABELS

    root.require_array(
        "band",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["band"],
        attributes={"long_name": "band name"},
    )
    root["band"][:] = ["snow_depth", "quality_flag"]

    root.require_array(
        "lat",
        shape=(NLAT,),
        chunks=(NLAT,),
        dtype="float32",
        dimension_names=["lat"],
        attributes={
            "units": "degrees_north",
            "long_name": "latitude",
        },
    )
    root["lat"][:] = lats

    root.require_array(
        "lon",
        shape=(NLON,),
        chunks=(NLON,),
        dtype="float32",
        dimension_names=["lon"],
        attributes={
            "units": "degrees_east",
            "long_name": "longitude",
        },
    )
    root["lon"][:] = lons

    root.require_array(
        "geophysical_data",
        shape=(0, 2, NLAT, NLON, 2),
        chunks=(1, 1, CLAT, NLON, 2),
        dtype="int16",
        fill_value=int(FILL_MISSING),
        compressors=[Zlib(level=9)],
        dimension_names=["time", "orbit", "lat", "lon", "band"],
        attributes={
            "long_name": (
                "geophysical data (band 0 = snow depth, "
                "band 1 = quality flag)"
            ),
            "units": "cm",
            "scale_factor": SCALE_FACTOR,
            "_FillValue": int(FILL_MISSING),
            "comment": (
                f"No observation value {int(FILL_NOOBS)} indicates ocean or "
                "areas without observation. This value is preserved in the "
                "raw data but is not treated as a fill value by CF conventions."
            ),
        },
    )

    session.commit(commit_msg)
    log.info("Initialized empty store")


def stored_days(repo: icechunk.Repository) -> list[int]:
    """Return stored day values in their current storage order."""
    session = repo.readonly_session("main")
    root = zarr.open_group(session.store, mode="r", zarr_format=3)
    return root["time"][:].tolist()


def insert_date(date: pd.Timestamp, repo: icechunk.Repository) -> None:
    """Insert a date using virtual references to the source HDF5 chunks."""
    day_val = int((date - TIME_EPOCH).days)
    date_iso = date.strftime("%Y-%m-%d")
    current = stored_days(repo)

    if day_val in current:
        slot = current.index(day_val)
        log.info("%s → skipped (already present, slot %d)", date_iso, slot)
        return

    orbit_manifests: dict[int, object] = {}
    for orbit_index, orbit in enumerate(ORBIT_CODES):
        chunk_url = gportal_url(date, orbit)
        try:
            manifest, _ = _get_manifests(chunk_url)
            orbit_manifests[orbit_index] = manifest
        except Exception as exc:
            log.warning(
                "  could not fetch %s for %s: %s",
                orbit,
                date_iso,
                exc,
            )

    if not orbit_manifests:
        log.info("%s → skipped (no orbits available on G-Portal)", date_iso)
        return

    new_slot = len(current)
    session = repo.writable_session("main")
    root = zarr.open_group(session.store, mode="r+", zarr_format=3)

    root["time"].resize((new_slot + 1,))
    root["time"][new_slot] = day_val
    root["geophysical_data"].resize((new_slot + 1, 2, NLAT, NLON, 2))

    # Each source chunk has shape (CLAT, NLON, 2), so both bands are
    # represented by the same virtual source chunk.
    for orbit_index, manifest in orbit_manifests.items():
        for (lat_chunk, lon_chunk, band_chunk), reference in manifest.iter_refs():
            key = (
                f"geophysical_data/c/{new_slot}/{orbit_index}/"
                f"{lat_chunk}/{lon_chunk}/{band_chunk}"
            )
            session.store.set_virtual_ref(
                key,
                reference["path"],
                offset=reference["offset"],
                length=reference["length"],
            )

    session.commit(f"Insert {date_iso} → slot {new_slot}")


def select_time_range(ds: xr.Dataset, start: str, end: str) -> xr.Dataset:
    """
    Select a time interval without requiring a sorted or complete time index.

    Dates are appended to the repository, so an existing repository may not
    have monotonically ordered time values. Boolean selection avoids pandas'
    restriction on slicing a non-monotonic DatetimeIndex.
    """
    times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
    mask = (times >= pd.Timestamp(start)) & (times <= pd.Timestamp(end))
    return ds.isel(time=np.flatnonzero(mask))


# Open or create the IceChunk store.
try:
    repo = icechunk.Repository.open(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        authorize_virtual_chunk_access={GPORTAL_URL: icechunk.credentials.HttpAccess},
    )
    log.info("Found existing icechunk store")
except icechunk.IcechunkError as exc:
    log.warning(
        "Failed to open existing repo with error: %s. "
        "Trying to create a fresh repo.",
        exc,
    )
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(GPORTAL_URL, icechunk.http_store())
    )
    repo = icechunk.Repository.create(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        config=config,
        authorize_virtual_chunk_access={GPORTAL_URL: icechunk.credentials.HttpAccess},
    )
    initialize_store(repo.writable_session("main"))


date_seq = pd.date_range(
    start="2018-09-01",
    end="2019-07-01",
    freq="D",
)

for date in tqdm(date_seq):
    insert_date(date, repo)


log.info("Testing reading a subset of data")
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store)

# Spatial, orbit, and band selection can use normal label selection. Time is
# selected with a boolean index because the repository may contain dates in
# insertion order rather than chronological order.
sub = ds.sel(lat=52.07, lon=-91.99, method="nearest")
sub = sub.sel(orbit="Descending", band="snow_depth")
sub = select_time_range(sub, "2018-10-01", "2019-02-01")

print(sub["geophysical_data"].values)

INFO:__main__:Found existing icechunk store
  0%|          | 1/304 [00:00<00:39,  7.73it/s]INFO:__main__:2018-09-02 → skipped (already present, slot 1)
INFO:__main__:2018-09-03 → skipped (already present, slot 2)
INFO:__main__:2018-09-04 → skipped (already present, slot 3)
  1%|▏         | 4/304 [00:00<00:17, 17.26it/s]INFO:__main__:2018-09-05 → skipped (already present, slot 4)
INFO:__main__:2018-09-06 → skipped (already present, slot 5)
INFO:__main__:2018-09-07 → skipped (already present, slot 6)
  2%|▏         | 7/304 [00:00<00:14, 20.53it/s]INFO:__main__:2018-09-08 → skipped (already present, slot 7)
INFO:__main__:2018-09-09 → skipped (already present, slot 8)
INFO:__main__:2018-09-10 → skipped (already present, slot 9)
  3%|▎         | 10/304 [00:00<00:13, 22.18it/s]INFO:__main__:2018-09-11 → skipped (already present, slot 10)
INFO:__main__:2018-09-12 → skipped (already present, slot 11)
INFO:__main__:2018-09-13 → skipped (already present, slot 12)
  4%|▍         | 13/304 [00:00<0

[ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.  27.9 59.4  0. ]


In [7]:
#!/usr/bin/env python

import os
# Silence icechunk rust warnings. Must be set before importing icechunk.
os.environ.setdefault("RUST_LOG", "error")

import icechunk
from virtualizarr.parsers.hdf.hdf import _construct_manifest_array

import obstore
from obstore.store import HTTPStore

import io
import logging
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from zarr.codecs import Zlib
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# Constants
NLAT, NLON = 1800, 3600          # HG grid dimensions
CLAT = 72                        # latitude chunk size (matches source HDF5)

# Updated fill values
FILL_MISSING = np.int16(-9999)   # fill value for missing retrievals
FILL_NOOBS = np.int16(-9999) 
#FILL_MISSING  = np.int16(-32768)   # cloudy / no retrieval
#FILL_NOOBS    = np.int16(-32767)   # ocean / no observation

SCALE_FACTOR = 0.1               # raw × 0.1 → cm

GPORTAL_BASE = (
    "https://gportal.jaxa.jp/download/standard/GCOM-W/GCOM-W.AMSR2"
    "/L3.SND_10/2/{yyyy}/{mm}/"
)
FNAME_TMPL = "GW1AM2_{date}_01D_{orbit}_L3SGSNDHG2210210.h5"
ORBIT_CODES = ["EQMA", "EQMD"]
ORBIT_LABELS = ["Ascending", "Descending"]

# CF time reference
TIME_UNITS = "days since 1970-01-01"
TIME_CALENDAR = "proleptic_gregorian"
TIME_EPOCH = pd.Timestamp("1970-01-01")

GPORTAL_URL = "https://gportal.jaxa.jp/"


def gportal_url(date: pd.Timestamp, orbit: str) -> str:
    """Return the G-Portal HTTPS URL for a date and orbit code."""
    return GPORTAL_BASE.format(
        yyyy=f"{date.year:04d}", mm=f"{date.month:02d}"
    ) + FNAME_TMPL.format(
        date=date.strftime("%Y%m%d"), orbit=orbit
    )


def _stream_hdf5(path: str, access_options: dict | None = None):
    """Return a context manager containing an HDF5 HTTP object in memory."""
    from urllib.parse import urlsplit

    if access_options is None:
        access_options = {}

    parts = urlsplit(path)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url, **access_options)
    data = obstore.get(store, obj_path).bytes()
    buf = io.BytesIO(bytes(data))

    class _CM:
        def __enter__(self):
            return buf

        def __exit__(self, *_):
            buf.close()

    return _CM()


def _av(value):
    """Convert an HDF5 attribute value to a JSON-serialisable value."""
    if isinstance(value, np.ndarray):
        flat = value.flatten()
        if flat.dtype.kind in ("S", "U", "O"):
            items = [
                item.decode() if isinstance(item, bytes) else str(item)
                for item in flat
            ]
            return items[0] if len(items) == 1 else items
        values = flat.tolist()
        return values[0] if len(values) == 1 else values
    return value.decode() if isinstance(value, bytes) else value


def _get_manifests(chunk_url: str) -> tuple:
    """Download an HDF5 file and extract its VirtualiZarr manifest."""
    from urllib.parse import urlsplit

    parts = urlsplit(chunk_url)
    base_url = f"{parts.scheme}://{parts.netloc}"
    obj_path = parts.path.lstrip("/")
    store = HTTPStore.from_url(base_url)
    buf = io.BytesIO(obstore.get(store, obj_path).bytes())

    try:
        with h5py.File(buf, "r") as hdf:
            geophysical_data = hdf["Geophysical Data"]
            manifest_array = _construct_manifest_array(
                chunk_url, geophysical_data, "/"
            )
            attrs = {
                key: _av(value)
                for key, value in geophysical_data.attrs.items()
            }
        return manifest_array.manifest, attrs
    finally:
        buf.close()


def initialize_store(
    session: icechunk.Session,
    commit_msg: str = "Initialize empty store",
) -> None:
    """Create the arrays in a new IceChunk repository."""
    lats = np.linspace(89.95, -89.95, NLAT).astype("float32")
    lons = np.linspace(0.05, 359.95, NLON).astype("float32")

    root = zarr.open_group(session.store, mode="w", zarr_format=3)

    root.require_array(
        "time",
        shape=(0,),
        chunks=(512,),
        dtype="int32",
        dimension_names=["time"],
        attributes={
            "units": TIME_UNITS,
            "calendar": TIME_CALENDAR,
            "long_name": "observation date",
        },
    )

    root.require_array(
        "orbit",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["orbit"],
        attributes={"long_name": "orbit direction"},
    )
    root["orbit"][:] = ORBIT_LABELS

    root.require_array(
        "band",
        shape=(2,),
        chunks=(2,),
        dtype=str,
        dimension_names=["band"],
        attributes={"long_name": "band name"},
    )
    root["band"][:] = ["snow_depth", "quality_flag"]

    root.require_array(
        "lat",
        shape=(NLAT,),
        chunks=(NLAT,),
        dtype="float32",
        dimension_names=["lat"],
        attributes={
            "units": "degrees_north",
            "long_name": "latitude",
        },
    )
    root["lat"][:] = lats

    root.require_array(
        "lon",
        shape=(NLON,),
        chunks=(NLON,),
        dtype="float32",
        dimension_names=["lon"],
        attributes={
            "units": "degrees_east",
            "long_name": "longitude",
        },
    )
    root["lon"][:] = lons

    root.require_array(
        "geophysical_data",
        shape=(0, 2, NLAT, NLON, 2),
        chunks=(1, 1, CLAT, NLON, 2),
        dtype="int16",
        fill_value=int(FILL_MISSING),
        compressors=[Zlib(level=9)],
        dimension_names=["time", "orbit", "lat", "lon", "band"],
        attributes={
            "long_name": (
                "geophysical data (band 0 = snow depth, "
                "band 1 = quality flag)"
            ),
            "units": "cm",
            "scale_factor": SCALE_FACTOR,
            "_FillValue": int(FILL_MISSING),
            "no_observation_value": int(FILL_NOOBS),
            "comment": (
                f"No observation value {int(FILL_NOOBS)} indicates ocean or "
                "areas without observation. This value is preserved in the "
                "raw data but is not treated as a fill value by CF conventions."
            ),
        },
    )

    session.commit(commit_msg)
    log.info("Initialized empty store")


def stored_days(repo: icechunk.Repository) -> list[int]:
    """Return stored day values in their current storage order."""
    session = repo.readonly_session("main")
    root = zarr.open_group(session.store, mode="r", zarr_format=3)
    return root["time"][:].tolist()


def insert_date(date: pd.Timestamp, repo: icechunk.Repository) -> None:
    """Insert a date using virtual references to the source HDF5 chunks."""
    day_val = int((date - TIME_EPOCH).days)
    date_iso = date.strftime("%Y-%m-%d")
    current = stored_days(repo)

    if day_val in current:
        slot = current.index(day_val)
        log.info("%s → skipped (already present, slot %d)", date_iso, slot)
        return

    orbit_manifests: dict[int, object] = {}
    for orbit_index, orbit in enumerate(ORBIT_CODES):
        chunk_url = gportal_url(date, orbit)
        try:
            manifest, _ = _get_manifests(chunk_url)
            orbit_manifests[orbit_index] = manifest
        except Exception as exc:
            log.warning(
                "  could not fetch %s for %s: %s",
                orbit,
                date_iso,
                exc,
            )

    if not orbit_manifests:
        log.info("%s → skipped (no orbits available on G-Portal)", date_iso)
        return

    new_slot = len(current)
    session = repo.writable_session("main")
    root = zarr.open_group(session.store, mode="r+", zarr_format=3)

    root["time"].resize((new_slot + 1,))
    root["time"][new_slot] = day_val
    root["geophysical_data"].resize((new_slot + 1, 2, NLAT, NLON, 2))

    # Each source chunk has shape (CLAT, NLON, 2), so both bands are
    # represented by the same virtual source chunk.
    for orbit_index, manifest in orbit_manifests.items():
        for (lat_chunk, lon_chunk, band_chunk), reference in manifest.iter_refs():
            key = (
                f"geophysical_data/c/{new_slot}/{orbit_index}/"
                f"{lat_chunk}/{lon_chunk}/{band_chunk}"
            )
            session.store.set_virtual_ref(
                key,
                reference["path"],
                offset=reference["offset"],
                length=reference["length"],
            )

    session.commit(f"Insert {date_iso} → slot {new_slot}")


def select_time_range(ds: xr.Dataset, start: str, end: str) -> xr.Dataset:
    """
    Select a time interval without requiring a sorted or complete time index.

    Dates are appended to the repository, so an existing repository may not
    have monotonically ordered time values. Boolean selection avoids pandas'
    restriction on slicing a non-monotonic DatetimeIndex.
    """
    times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
    mask = (times >= pd.Timestamp(start)) & (times <= pd.Timestamp(end))
    return ds.isel(time=np.flatnonzero(mask))

# Open or create the IceChunk store.
try:
    repo = icechunk.Repository.open(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        authorize_virtual_chunk_access={GPORTAL_URL: icechunk.credentials.HttpAccess},
    )
    log.info("Found existing icechunk store")
except icechunk.IcechunkError as exc:
    log.warning(
        "Failed to open existing repo with error: %s. "
        "Trying to create a fresh repo.",
        exc,
    )
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(GPORTAL_URL, icechunk.http_store())
    )
    repo = icechunk.Repository.create(
        icechunk.s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND",
        ),
        config=config,
        authorize_virtual_chunk_access={GPORTAL_URL: icechunk.credentials.HttpAccess},
    )
    initialize_store(repo.writable_session("main"))


date_seq = pd.date_range(
    start="2018-09-01",
    end="2019-07-01",
    freq="D",
)

for date in tqdm(date_seq):
    insert_date(date, repo)


log.info("Testing reading a subset of data")
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store)

ds['geophysical_data'] = ds['geophysical_data'].fillna(FILL_MISSING)

# Spatial, orbit, and band selection can use normal label selection. Time is
# selected with a boolean index because the repository may contain dates in
# insertion order rather than chronological order.
#sub = ds.sel(lat=52.07, lon=-91.99, method="nearest").sel(orbit="Descending", band="snow_depth")
sub = ds.sel(lat=0., lon=-91.99, method="nearest").sel(orbit="Descending", band="snow_depth")
sub = select_time_range(sub, "2018-10-01", "2019-02-01")

print(sub["geophysical_data"].values)

INFO:__main__:Found existing icechunk store
  0%|          | 1/304 [00:00<00:40,  7.45it/s]INFO:__main__:2018-09-02 → skipped (already present, slot 1)
INFO:__main__:2018-09-03 → skipped (already present, slot 2)
  1%|          | 3/304 [00:00<00:27, 10.94it/s]INFO:__main__:2018-09-04 → skipped (already present, slot 3)
INFO:__main__:2018-09-05 → skipped (already present, slot 4)
  2%|▏         | 5/304 [00:00<00:22, 13.08it/s]INFO:__main__:2018-09-06 → skipped (already present, slot 5)
INFO:__main__:2018-09-07 → skipped (already present, slot 6)
  2%|▏         | 7/304 [00:00<00:20, 14.54it/s]INFO:__main__:2018-09-08 → skipped (already present, slot 7)
INFO:__main__:2018-09-09 → skipped (already present, slot 8)
  3%|▎         | 9/304 [00:00<00:19, 15.33it/s]INFO:__main__:2018-09-10 → skipped (already present, slot 9)
INFO:__main__:2018-09-11 → skipped (already present, slot 10)
  4%|▎         | 11/304 [00:00<00:18, 16.01it/s]INFO:__main__:2018-09-12 → skipped (already present, slot 11)


[-9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999. -9999.
 -9999. -9999. -9999. -9999.]


In [11]:
print(ds)
print(ds['geophysical_data'].attrs)

for key, value in ds['geophysical_data'].attrs.items():
    print(f"{key}: {value}")

<xarray.Dataset> Size: 113GB
Dimensions:           (time: 547, orbit: 2, lat: 1800, lon: 3600, band: 2)
Coordinates:
  * time              (time) datetime64[ns] 4kB 2018-09-01 ... 2018-08-31
  * orbit             (orbit) object 16B 'Ascending' 'Descending'
  * lat               (lat) float32 7kB 89.95 89.85 89.75 ... -89.85 -89.95
  * lon               (lon) float32 14kB 0.05 0.15 0.25 ... 359.8 359.9 360.0
  * band              (band) object 16B 'snow_depth' 'quality_flag'
Data variables:
    geophysical_data  (time, orbit, lat, lon, band) float64 113GB dask.array<chunksize=(1, 1, 72, 3600, 2), meta=np.ndarray>
{'long_name': 'geophysical data (band 0 = snow depth, band 1 = quality flag)', 'units': 'cm'}
long_name: geophysical data (band 0 = snow depth, band 1 = quality flag)
units: cm


In [12]:
print(ds.info())

xarray.Dataset {
dimensions:
	time = 547 ;
	orbit = 2 ;
	lat = 1800 ;
	lon = 3600 ;
	band = 2 ;

variables:
	float64 geophysical_data(time, orbit, lat, lon, band) ;
		geophysical_data:long_name = geophysical data (band 0 = snow depth, band 1 = quality flag) ;
		geophysical_data:units = cm ;
	object band(band) ;
		band:long_name = band name ;
	float32 lat(lat) ;
		lat:units = degrees_north ;
		lat:long_name = latitude ;
	float32 lon(lon) ;
		lon:units = degrees_east ;
		lon:long_name = longitude ;
	object orbit(orbit) ;
		orbit:long_name = orbit direction ;
	datetime64[ns] time(time) ;
		time:long_name = observation date ;

// global attributes:
}None
